In [1]:
import mdtraj as md
import os
import glob
import pandas as pd

In [2]:
output_path='/global/cfs/cdirs/m4704/100125_Nature_Com_data/single_conformation/nmr'

In [3]:
ref_path='/global/cfs/cdirs/m4704/100125_Nature_Com_data/Apo_holo_data/pdbs'

In [4]:
def generate_rmsd_alphaSAXS(pdb_id,ref_path,output_path):
    rmsd=[]
    ref_t=md.load(os.path.join(ref_path,pdb_id+'.pdb'))
    ref_atom=ref_t.topology.select('name CA')
    test_t = md.load(os.path.join(output_path, pdb_id+'.pdb'))
    test_atom=test_t.topology.select('name CA')
    if len(test_atom)!=len(ref_atom):
        return pdb_id+' has the wrong length'
    ca_selection=md.rmsd(test_t,ref_t,0,test_atom,ref_atom)
    return ca_selection[0]*10

In [5]:
pdb_list=[ f[:-4] for f in os.listdir(ref_path) if f.endswith('.pdb')]

In [6]:
len(pdb_list)

80

In [7]:
result={}

In [8]:
for i in pdb_list:
    try:
        result[i]=generate_rmsd_alphaSAXS(i,ref_path,output_path)
    except:
        print(f'{i} is not in the prediction dataset')

In [9]:
result

{'2K8R-1_A': 3.7439680099487305,
 '1XSA-23_A': 4.088649153709412,
 '1K2H-4_A': 6.395006775856018,
 '2D9E-12_A': 3.590703010559082,
 '1P7M-18_A': 2.488432079553604,
 '1EX6_B': 3.0724433064460754,
 '2AI6-18_A': 2.787221372127533,
 '1MO8-14_A': 18.085795640945435,
 '2KXL-8_A': 3.8860225677490234,
 '1TJD_A': 1.699306070804596,
 '2VCD-8_A': 1.5791341662406921,
 '2F63-4_A': 2.2038356959819794,
 '1ROE-10_A': 4.396737217903137,
 '1CZ2-8_A': 2.4383819103240967,
 '1PUN-7_A': 3.8825681805610657,
 '1WCW_A': 1.5908023715019226,
 '1ZFS-13_B': 2.730790376663208,
 '1VHL_A': 2.1964825689792633,
 '2LKC-4_A': 7.930576801300049,
 '1ZOL_A': 2.4884289503097534,
 '1NTR-7_A': 4.7535717487335205,
 '1UR6-4_A': 2.3095758259296417,
 '1TNQ-33_A': 5.227321982383728,
 '1S2O_A': 2.2988244891166687,
 '1JM4-15_B': 4.142813682556152,
 '1RRO_A': 1.1153163015842438,
 '2CG6_A': 7.207493782043457,
 '1MO7-3_A': 18.58777642250061,
 '1VIY_C': 2.35550194978714,
 '1RTP_1': 1.0937371850013733,
 '1JKN-15_A': 3.819941282272339,
 '1

In [10]:
df = pd.DataFrame(result.items(), columns=['Key', 'Value'])

In [11]:
df.describe()

,Value
count,80.000000
mean,4.541702
std,4.146548
min,1.093737
25%,2.336225
50%,3.348509
75%,4.470605
max,18.923018


In [11]:
pair_csv=pd.read_csv('/global/cfs/cdirs/m4704/100125_Nature_Com_data/Apo_holo_data/Table_rmsd_Apo_vs_Holo.csv',sep=';')

In [12]:
pair_dict={}
for index, i in pair_csv.iterrows():
    pair_dict[i['Apo_ID']]=i['Holo_ID']
    #pair_dict[i['Holo_ID']]=i['Apo_ID']

In [13]:
result_csv=pd.DataFrame.from_dict(result,orient='index').reset_index().rename(columns={'index':'ID',0:'RMSD'})

In [14]:
result_csv['pair_ID']=result_csv['ID'].map(pair_dict)

In [15]:
result_pair=result_csv.dropna().merge(result_csv[['ID',
'RMSD']],left_on='pair_ID',right_on='ID',suffixes=['','_pair']).drop(columns={'pair_ID'})

In [16]:
result_pair

,ID,RMSD,ID_pair,RMSD_pair
0,1XSA-23_A,4.088649,1XSC-20_A,2.449033
1,1K2H-4_A,6.395007,1ZFS-13_B,2.730790
2,2D9E-12_A,3.590703,2RS9-13_B,3.332593
3,1EX6_B,3.072443,1EX7_A,2.095554
4,2AI6-18_A,2.787221,2OZW-3_A,3.629375
5,2KXL-8_A,3.886023,2K0G-9_A,3.428072
6,1TJD_A,1.699306,1EEJ_B,2.047089
7,2F63-4_A,2.203836,1EQM_A,3.364425
8,2LKC-4_A,7.930577,2LKD-18_A,5.965340
9,1ZOL_A,2.488429,1O03_A,2.401163


In [19]:
AlphaFold_df=pd.read_csv('AlphaFold_RMSD.csv',index_col='ID')
AlphaFold_df=AlphaFold_df.drop(AlphaFold_df.columns[0], axis=1)

In [20]:
AlphaFold_df['RMSD_Fold'] = pd.to_numeric(AlphaFold_df['RMSD_Fold'], errors='coerce')

In [21]:
AlphaFold_df=AlphaFold_df.dropna()

In [22]:
AlphaFold_dict=AlphaFold_df.to_dict()['RMSD_Fold']

In [23]:
result_pair.dtypes

ID            object
RMSD         float64
ID_pair       object
RMSD_pair    float64
dtype: object

In [24]:
AlphaFold_df.dtypes

RMSD_Fold    float64
dtype: object

In [25]:
for index, i in result_pair.iterrows():
    result_pair.loc[index,'RMSD']-=AlphaFold_dict[i['ID']]
    result_pair.loc[index,'RMSD_pair']-=AlphaFold_dict[i['ID_pair']]

In [28]:
result_pair['RMSD_sum']=result_pair['RMSD']+result_pair['RMSD_pair']

In [32]:
result_pair.sort_values('RMSD_sum')

,ID,RMSD,ID_pair,RMSD_pair,RMSD_sum
15,1JFJ-3_A,-6.162395,1JFK_A,-2.109873,-8.272268
28,4AKE_B,-2.313638,2ECK_B,-4.337800,-6.651437
13,1MO7-3_A,-2.238847,1MO8-14_A,-1.583627,-3.822474
6,1TJD_A,-0.797425,1EEJ_B,-0.969317,-1.766742
25,1FMF-4_A,-0.902580,1ID8-11_A,-0.568527,-1.471107
14,1VIY_C,-0.440276,1VHL_A,-0.355732,-0.796008
16,1LIP-2_A,-0.450644,1JTB-6_A,-0.233744,-0.684389
23,1WD7_B,-0.426444,1WCW_A,-0.177141,-0.603586
29,1GH1-7_A,-0.236910,1CZ2-8_A,-0.257443,-0.494353
0,1XSA-23_A,0.175216,1XSC-20_A,-0.599066,-0.423850
